# 2.2-Common QC

**Inés Sentís**

Date of execution

In [ ]:
Sys.Date()

## Libraries

In [ ]:
suppressMessages(suppressWarnings({
library(Seurat)
library(SeuratDisk)
library(Matrix)
library(tidyverse)
library(ggpubr)
library(readr)
library(dplyr)
library(ggplot2)
library(here)
library(glue)
}))

## Parameters

In [ ]:
args <- commandArgs(TRUE)
subproj <- args[2]

source(here(glue("{subproj}/st_analysis/misc/paths.R")))
source(here("utils/bin.R"))

## Load data

We have 4 different samples each one from a different patient that we are going to analyze separately.

In [ ]:
options(warn = -1, verbose = FALSE)
id_sp_df <- read_csv(
  file = here(glue("{spaceranger}/data/sample_id.csv")),
  show_col_types = FALSE
)

sp_ls <- lapply(id_sp_df$gem_id, function(gem_id) {
  spaceranger_data <- glue("{spaceranger}/results/{gem_id}/outs")
  
  sample_id <- id_sp_df[id_sp_df$gem_id == gem_id, ] %>%
    pull(patient)
  
  # load spatial data
  counts <- Read10X_h5(filename = here(spaceranger_data, "filtered_feature_bc_matrix.h5"))
  
  # add patient/sample prefix to barcodes
  colnames(counts) <- paste0(sample_id, "_", colnames(counts))
  
  # build Seurat object with counts + image
  se_obj <- CreateSeuratObject(
    counts = counts,
    assay = "Spatial",
    project = sample_id
  )
  
  # attach spatial image
  image_obj <- Load10X_Spatial(data.dir = here(glue("{spaceranger}/results/{gem_id}/outs")), 
                                     filename='filtered_feature_bc_matrix.h5')
  names(image_obj@images) <- sample_id
  image_obj@images[[sample_id]]@key <- paste0(sample_id, "_")
  
  se_obj@images <- image_obj@images
  
  # Full coordinates object
  coords <- se_obj@images[[sample_id]]@coordinates

  # Update rownames
  rownames(coords) <- paste(sample_id, rownames(coords), sep = "_")

  # Assign it back
  se_obj@images[[sample_id]]@coordinates <- coords
  
  # add metadata
  se_obj[["gem_id"]] <- gem_id
  se_obj[["sample_id"]] <- sample_id
  se_obj[["patient"]] <- sample_id  
  
  return(se_obj)
})

In [ ]:
# now merge (no need for add.cell.ids anymore)
se_obj <- merge(
  sp_ls[[1]],
  y = sp_ls[2:length(sp_ls)],
  project = subproj
)

Add mitochondrial and ribosomal %

In [ ]:
# Collect all genes coded on the mitochondrial genome
se_obj[["percent.mito"]] <- PercentageFeatureSet(
  object = se_obj,
  pattern = "^MT-")
summary(se_obj[["percent.mito"]])

# Collect all genes coding for ribosomal proteins
se_obj[["percent.ribo"]] <- PercentageFeatureSet(
  object = se_obj,
  pattern = "^RPL|^RPS")
summary(se_obj[["percent.ribo"]])

## QC Analysis

### Remove empty genes

We start by removing those genes that aren't expressed in any of the spots overlaying the tissue

In [ ]:
table(rowSums(as.matrix(se_obj@assays$Spatial@counts)) == 0)

keep_genes <- rowSums(as.matrix(se_obj@assays$Spatial@counts)) != 0
se_obj <- se_obj[keep_genes, ]

### Basic features
#### Number of genes

We start by plotting some basic features that will help us visualize and define filtering options.
We start by plotting the number of genes per spot, *complexity*, to assess if there are empty spots or huge disparity.

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10, verbose = FALSE)
p1 <- ggplot() +
  geom_histogram(data = se_obj[[]], 
                          aes(nFeature_Spatial),
                          fill = "red",
                          alpha = 0.7,
                          color = "red",
                          bins = 50) +
  facet_wrap(. ~ patient, scales = "free") +
  ggtitle("Unique genes per spot") +
  labs(x = "Number of Detected Genes",
                y = "Number of Spots") +
  theme_pubr()

p1

After looking at the distribution we are also going to look at how these spots look on the tissue

In [ ]:
lapply(id_sp_df$patient, function(pat) {
    
  SpatialFeaturePlot(
    object = se_obj[, se_obj$patient == pat],
    features = "nFeature_Spatial",
    images = glue("{pat}")) +
    labs(title = pat) +
    theme(plot.title = element_text(hjust = 0.5, size = 18),
                   legend.text = element_text(angle = 45, hjust = 1))
  
}) %>% cowplot::plot_grid(
  plotlist = .,
  align = "hv",
  axis = "trbl")

### Gene counts

Another characteristic we want to look at is how many counts per gene there are since we want to remove lowly expressed genes which aren't giving information but potentially introducing noise.

In [ ]:
p2 <- ggplot() +
  geom_histogram(data = se_obj[[]],
                          aes(nCount_Spatial),
                          fill = "red",
                          alpha = 0.7,
                          color = "red",
                          bins = 50) +
  facet_wrap(. ~ patient, scales = "free") +
  ggtitle("Total counts per spots") +
  labs(x = "Library Size (total UMI)",
                y = "Number of Spots") +
  theme_pubr()

p2

In [ ]:
count_mtrx <- GetAssayData(object = se_obj,
                                   slot = "counts",
                                   assay = "Spatial")

gene_attr <- lapply(id_sp_df$patient, function(pat) {
  mask <- str_detect(
    string = colnames(count_mtrx),
    pattern = pat)
  
  gene_attr <- data.frame(
    nUMI = rowSums(count_mtrx[, mask]), 
    nSpots = rowSums(count_mtrx[, mask] > 0),
    patient = pat)
}) %>%
  bind_rows()

p3 <- ggplot() +
  geom_histogram(data = gene_attr,
                          aes(nUMI),
                          fill = "red",
                          alpha = 0.7,
                          color = "red",
                          bins = 50) +
  facet_wrap(. ~ patient, scales = "free") +
  scale_x_log10() +
  ggtitle("Total counts per gene (log10 scale)") +
  theme_pubr()
p3

#### Gene ubiquitousness
We also look at on how many spots each gene is detected, we see there are a few genes expressed in almost all the spots while and a majority of genes detected in few spots.

In [ ]:
p4 <- ggplot() +
  geom_histogram(data = gene_attr,
                          aes(nSpots),
                          fill = "red",
                          alpha = 0.7,
                          color = "red",
                          bins = 50) +
  facet_wrap(. ~ patient, scales = "free") +
  ggtitle("Total spots per gene") +
  theme_pubr()

p4

#### Mitochondrial %

Next we take a look at the mitochondrial %; This metric can help us get a first glimpse of metabolic activity and/or necrotic regions - 10X explains [here](https://kb.10xgenomics.com/hc/en-us/articles/360001086611-Why-do-I-see-a-high-level-of-mitochondrial-gene-expression-)

In [ ]:
p5 <- ggplot() +
  geom_histogram(data = se_obj[[]], 
                          aes(percent.mito),
                          fill = "red",
                          alpha = 0.7,
                          color = "red",
                          bins = 50) +
  facet_wrap(. ~ patient, scales = "free") +
  ggtitle("Mitochondrial % per spot") +
  labs(x = "Mitochondrial % ",
                y = "Number of Spots") +
  theme_pubr()
p5

In [ ]:
lapply(id_sp_df$patient, function(pat) {
  
  SpatialFeaturePlot(
    object = se_obj[, se_obj$patient == pat],
    features = "percent.mito",
    images = glue("{pat}")) +
    labs(title = pat) +
    theme(plot.title = element_text(hjust = 0.5, size = 18))
  
}) %>% cowplot::plot_grid(plotlist = .)

#### Ribosomal %
Lastly we look at the ribosomal % which gives us insight into which regions are the most transcriptomically active when looked side by side with the number of detected genes.

In [ ]:
p6 <- ggplot() +
  geom_histogram(data = se_obj[[]], 
                          aes(percent.ribo),
                          fill = "red",
                          alpha = 0.7,
                          color = "red",
                          bins = 50) +
  facet_wrap(. ~ patient, scales = "free") +
  ggtitle("Ribosomal % per spot") +
  labs(x = "Ribosomal % ",
                y = "Number of Spots") +
  theme_pubr()
p6

In [ ]:
lapply(id_sp_df$patient, function(pat) {
  
  SpatialFeaturePlot(
    object = se_obj[, se_obj$patient == pat],
    features = "percent.ribo",
    images = glue("{pat}")) +
    labs(title = pat) +
    theme(plot.title = element_text(hjust = 0.5, size = 18))
  
}) %>% cowplot::plot_grid(plotlist = .)

### Feature covariation
Next we look at how these features covariate.

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 20, verbose = FALSE)
plt_covar_ls <- qc_covar_plots(se = se_obj,
                               nfeat = "nFeature_Spatial",
                               ncount = "nCount_Spatial",
                               slot = "counts",
                               assay = "Spatial",
                               percent.mito = "percent.mito",
                               percent.ribo = "percent.ribo",
                               facet = "patient")

cowplot::plot_grid(plotlist = plt_covar_ls, ncol = 2, align = "hv", axis = "trbl")

### Check image miss alignment

In [ ]:
stutility_plts <- lapply(id_sp_df$gem_id, function(id) {
  sample_id <- id_sp_df %>% filter(gem_id == id) %>% pull(patient) 
  proj_dir <- here()
  spaceranger_data <- glue("{proj_dir}/{spaceranger}/results/{id}/outs")
  # The recommended method to read the files into R is via the creation of a data.frame that we will call the infoTable.
  # There are four columns in this table that are required for Visium data: "samples", "spotfiles", "imgs" and "json".
  # These columns specify the paths to the required input files. 
  infoTable <- data.frame(
    samples = glue("{spaceranger_data}/raw_feature_bc_matrix.h5"),
    spotfiles = glue("{spaceranger_data}/spatial/tissue_positions_list.csv"),
    imgs = glue("{spaceranger_data}/spatial/tissue_hires_image.png"),
    json = glue("{spaceranger_data}/spatial/scalefactors_json.json")
  )
  # Load data
  se <- STutility::InputFromTable(infoTable, disable.subset = TRUE)
  # Collect all genes coded on the mitochondrial genome
  mt.genes <- grep(pattern = "^MT-", x = rownames(se), value = TRUE)
  se$percent.mito <- 
    (colSums(se@assays$RNA@counts[mt.genes, ]) / 
       colSums(se@assays$RNA@counts)) * 100
  # Collect all genes coding for ribosomal proteins
  rp.genes <- grep(pattern = "^RPL|^RPS", x = rownames(se), value = TRUE)
  se$percent.ribo <- 
    (colSums(se@assays$RNA@counts[rp.genes, ]) / 
       colSums(se@assays$RNA@counts)) * 100
  se <- STutility::LoadImages(se, time.resolve = F, verbose = T)
  

  p<-STutility::FeatureOverlay(
    se, 
    features = c("nFeature_RNA", "nCount_RNA",
                 "percent.ribo", "percent.mito"), 
    pt.size = 1.2,
    cols = c("lightgrey", "yellow", "orange", "red", "dark red"), 
    dark.theme = FALSE, 
    type = "raw",
    pt.alpha = 1,
    ncols = 2) +
    patchwork::plot_annotation(title = glue("Sample ID/patient - {sample_id}")) &
    coord_fixed(ratio = 1)
  
  ggsave(plot=p, width = 15, height = 10, dpi = 300, 
       file = here(glue("{qc}/{plt_dir}/{sample_id}_qc_stutility.png")))
  p
})

In [ ]:
stutility_plts

## Save

Save the object to use downstream.

In [ ]:
lapply(id_sp_df$patient, function(pat) {
  
  se_obj_sub <- se_obj[, se_obj$patient == pat]
  
  # Remove other images
  img <- pat
  se_obj_sub@images <- se_obj_sub@images[Images(se_obj_sub) == img]
  
  "{qc}/{robj_dir}/2.2-qc_se_{pat}.rds" %>%
    glue() %>%
    here() %>%
    saveRDS(
      object = se_obj_sub,
      file = .)
})

In [ ]:
sessionInfo()